In [160]:
import dotenv

dotenv.load_dotenv()

True

In [136]:
import os

openrouter_key = os.getenv("OPENROUTER_KEY")

In [137]:
# from openai import OpenAI

# client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=openrouter_key,
# )

# # First API call with reasoning
# response = client.chat.completions.create(
#     model="openai/gpt-oss-20b:free",
#     messages=[
#         {"role": "user", "content": "How many r's are in the word 'strawberry'?"}
#     ],
#     extra_body={"reasoning": {"enabled": True}},
# )

# # Extract the assistant message with reasoning_details
# response = response.choices[0].message

In [138]:
# print(response)

In [139]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key,
)

In [147]:
TOPIC_PROMPT = """
일본어 라디오에서 사용할 주제를 1개 생성하라.

[조건]
- JLPT {LEVEL} 청취자가 이해할 수 있는 쉬운 주제
- 일상적이고 공감 가능한 내용
- 너무 추상적이거나 어려운 개념 금지

[출력 규칙]
- 오직 "주제 한 줄"만 출력
- 불필요한 설명, 따옴표, 번호, 줄바꿈 금지
- 20자 이내로 작성

[중복 방지]
- 아래 주제들과 의미적으로 겹치지 않도록 할 것:
{PREVIOUS_TOPICS}
"""
SCRIPT_PROMPT = """
{TOPIC}
이 주제를 기반으로 JLPT {LEVEL} 수준 청취자를 위한 일본어 라디오 스크립트를 작성하라.

[목표]
- 일본어 초급 학습자가 듣고 이해할 수 있는 자연스러운 라디오

[프로그램 정보]
- 프로그램 이름: ゆるっと電波 {LEVEL}
- 진행자 이름: ハヤト

[언어 수준]
- JLPT {LEVEL} 수준의 어휘와 문법을 우선 사용
- 불가피하게 어려운 표현이 포함될 경우 최대 5개 이하로 제한
- 어려운 단어는 가능한 쉬운 표현으로 바꿈

[스타일]
- 캐주얼하고 부드러운 말투 사용 (친근한 라디오 진행자 느낌)
- 청자에게 말을 거는 표현 포함 (예: みなさん、どうですか？)
- 딱딱한 설명체 금지

[문장 구조]
- 문장은 짧고 유기적으로 작성
- 문장을 과도하게 길게 작성하지 않음(한 문장 70자 이내)
- 자연스러운 호흡을 위해 「、」「。」 적절히 사용

[청해 최적화]
- 발음하기 어려운 한자, 언어 수준에 맞지 않은 한자, 외래어, 숫자는 가능한 한 풀어서 표현
- 의미 단위로 끊어 읽기 쉽게 구성

[재미 요소]
- 가벼운 감정 표현 또는 공감 요소 포함
- 청취자가 상황을 상상할 수 있도록 묘사 추가

[출력 형식]
- 일본어 라디오 대본 형태로 출력
- 불필요한 설명 없이 대본만 출력
- 대본은 オープニング,セグメント,コーナー,エンディング 섹터로 구성

[オープニング]
- 자기소개와 프로그램 소개 간단히 포함

[セグメント]
- 총 3개로 구성, 섹터 제목은 동일하게 [セグメント]로 할 것
- 라디오의 주제와 관련된 간단한 이야기를 포함

[コーナー]
- 라디오에서 사용된 핵심 일본어 표현 2~3개 설명
- 각 표현의 의미와 사용 상황을 간단히 설명
- 비슷한 쉬운 표현 1개 추가 소개 (선택)
- 핵심 문장 1개를 제시하고 따라 말해볼 수 있도록 유도

[エンディング]
- 청취자가 자신의 경험과 연결해 생각해볼 수 있는 질문 1개 추가
- 마무리 인사
"""

REWRITE_TTS_PROMPT = """
{SCRIPT}
이 대본을 TTS용 일본어 라디오 스크립트로 재작성하라.

조건:
1. JLPT {LEVEL} 수준 청취자가 이해 가능해야 한다
2. 발음이 자연스럽도록 어려운 한자, 영어, 숫자를 풀어쓴다
3. 문장은 짧고 말하듯이 유기적으로 작성한다 (한 문장 70자 이내)
4. 쉼표(、)와 마침표(。)를 사용해 호흡을 만든다
5. 라디오 진행자처럼 부드럽고 따뜻한 말투로 작성한다
6. 청자에게 말을 거는 표현을 포함한다
7. 청자의 반응을 묘사하지 않는다 (예: みんなで繰り返す)
8. 감정 표현을 자연스럽게 포함한다
9. [オープニング][セグメント][コーナー][エンディング] 형식으로 섹터를 구분한다
10. 섹터 구분에 반드시 대괄호를 사용하며(예: [オープニング]), 이외에는 대괄호를 사용하지 않는다
11. 자연스러운 히라가나 웃음 발화 표현을 반드시 사용하되, 과도한 사용은 피한다
12. 오로지 진행자의 대사 또는 진행자 목소리를 통한 인용만 허용한다
13. オープニング의 첫 대사는 반드시 "みなさん、こんにちは！「ゆるっと電波 {LEVEL}」へようこそ！私はハヤトです。"로 시작한다
14. 대본은 TTS로 읽었을 때 자연스럽게 들리도록 작성한다
15. 발화자를 스크립트에 직접 명시하는 형식을 사용하지 않는다
16. 결과는 일반 텍스트로만 출력하며, 코드 블록( ```)이나 포맷팅 문법(** 등)을 절대 사용하지 않는다
17. 출력은 바로 읽을 수 있는 라디오 대본 형태로만 제공한다 (추가 설명 금지)
"""

In [ ]:
import json
from pathlib import Path


class ScriptManager:
    base_path: Path
    topics: list[str]

    def __init__(self, base_path: Path):
        self.base_path = base_path
        self.topics = self.load_topics()

    def load_topics(self):
        topics_file = self.base_path / "topics.json"

        if topics_file.exists():
            with open(topics_file, "r", encoding="utf-8") as f:
                return json.load(f)

        return []

    def save_topics(self):
        topics_file = self.base_path / "topics.json"

        with open(topics_file, "w", encoding="utf-8") as f:
            json.dump(self.topics, f, indent=2, ensure_ascii=False)

    def new_topic(self, topic: str):
        self.topics.append(topic)
        self.save_topics()

    def save_script(self, script, topic, now):
        scripts_dir = self.base_path / "scripts"
        if not scripts_dir.exists():
            scripts_dir.mkdir(parents=True)

        data = {"time": now, "topic": topic, "script": script}

        filename = os.path.join(scripts_dir, f"{now}.json")

        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
from langgraph.graph import StateGraph
from typing import TypedDict, Optional, Callable
from pathlib import Path
import json
from datetime import datetime
from pydub import AudioSegment
import re


class RadioState(TypedDict):
    topic: Optional[str]
    script: Optional[str]
    tts_script: Optional[str]
    audio_path: Optional[str]
    now: Optional[str]


class AudioManager:
    client: OpenAI
    base_path: Path

    def __init__(self, base_path: Path):
        self.base_path = base_path
        self.client = OpenAI()

    def preprocess_for_tts(self, script):
        script = script.replace("---", "")
        script = re.sub(r"\[.*?\]", "。。。\n", script)
        script += "。\n"

        return script

    def save_tts(self, script, now):
        if not self.base_path.joinpath("tts").exists():
            self.base_path.joinpath("tts").mkdir(parents=True)

        output_path = self.base_path / "tts" / f"{now}.mp3"

        response = self.client.audio.speech.create(
            model="gpt-4o-transcribe",
            voice="alloy",  # or "verse", "aria"
            speed=1.0,
            input=script,
        )

        with open(output_path, "wb") as f:
            f.write(response.content)

        # print(f"🎧 audio file saved: {output_path}")

        return output_path

    def mix_bgm(self, now):
        if not self.base_path.joinpath("audio").exists():
            self.base_path.joinpath("audio").mkdir(parents=True)

        tts_path = self.base_path / "tts" / f"{now}.mp3"
        bgm_path = self.base_path / "bgm.mp3"
        output_path = self.base_path / "audio" / f"{now}.mp3"

        voice = AudioSegment.from_file(tts_path)
        bgm = AudioSegment.from_file(bgm_path)

        # BGM 길이를 음성에 맞춤 (loop)
        if len(bgm) < len(voice):
            times = len(voice) // len(bgm) + 1
            bgm = bgm * times

        bgm = bgm[: len(voice)]

        bgm = bgm - 20  # dB 줄임

        # 합치기
        mixed = voice.overlay(bgm)

        mixed.export(output_path, format="mp3")

        # print(f"🎶 bgm is mixed: {output_path}")
        return output_path


class Radiograph(StateGraph[RadioState]):
    client: OpenAI
    is_debug: bool

    level: str

    TOPIC_PROMPT: str
    SCRIPT_PROMPT: str
    REWRITE_TTS_PROMPT: str

    script_manager: ScriptManager
    audio_manager: AudioManager

    def __init__(
        self,
        client: OpenAI,
        base_path: Path,
        level: str,
        sectors: list[str],
        TOPIC_PROMPT: str,
        SCRIPT_PROMPT: str,
        REWRITE_TTS_PROMPT: str,
        is_debug: bool = False,
    ):
        super().__init__(
            state_schema=RadioState,
            name="Radiograph",
        )

        if not base_path.exists():
            base_path.mkdir(parents=True)

        self.script_manager = ScriptManager(base_path=base_path)
        self.audio_manager = AudioManager(base_path=base_path)

        self.client = client
        self.is_debug = is_debug

        self.level = level
        self.sectors = sectors

        self.TOPIC_PROMPT = TOPIC_PROMPT
        self.SCRIPT_PROMPT = SCRIPT_PROMPT
        self.REWRITE_TTS_PROMPT = REWRITE_TTS_PROMPT

        self.add_node("topic", self.topic_node)
        self.add_node("script", self.script_node)
        self.add_node("rewrite_tts", self.rewrite_tts_node)
        self.add_node("tts", self.tts_node)

        self.set_entry_point("topic")

        self.add_edge("topic", "script")
        self.add_edge("script", "rewrite_tts")
        self.add_edge("rewrite_tts", "tts")

    def debug(self, message: str):
        if self.is_debug:
            print(message)

    def run_prompt(
        self,
        prompt: str,
        validation: Callable[[str], bool] = None,
        max_retries: int = 3,
    ):
        for attempt in range(max_retries):
            response = self.client.chat.completions.create(
                model="openai/gpt-oss-20b:free",
                messages=[{"role": "user", "content": prompt}],
                extra_body={"reasoning": {"enabled": True}},
            )
            self.debug(f"response: {response}")

            message = response.choices[0].message.content.strip()

            if not validation or validation(message):
                break

            self.debug(f"Validation failed for attempt {attempt + 1}. Retrying...")

        return message

    def test_sectors(self, tts_script: str):
        for sector in self.sectors:
            if f"[{sector}]" not in tts_script:
                return False
        return True

    def replace_sectors(self, tts_script: str):
        tts_space = "------"
        for sector in self.sectors:
            tts_script = tts_script.replace(f"[{sector}]", tts_space)
        return tts_script

    def topic_node(self, state: RadioState):
        previous_topics = self.script_manager.load_topics()

        self.debug(f"previous_topics: {previous_topics}")

        prompt = self.TOPIC_PROMPT.replace("{LEVEL}", self.level).replace(
            "{PREVIOUS_TOPICS}", json.dumps(previous_topics, ensure_ascii=False)
        )

        self.debug(f"topic_node prompt: {prompt}")

        topic = self.run_prompt(prompt)

        self.script_manager.new_topic(topic)
        state["topic"] = topic

        self.debug(f"generated topic: {topic}")

        return state

    def script_node(self, state: RadioState):
        self.debug(f"topic: {state['topic']}")

        prompt = self.SCRIPT_PROMPT.replace("{LEVEL}", self.level).replace(
            "{TOPIC}", state["topic"]
        )

        self.debug(f"script_node prompt: {prompt}")

        script = self.run_prompt(prompt)

        now = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.script_manager.save_script(script, state["topic"], now)

        state["now"] = now
        state["script"] = script

        self.debug(f"generated script: {script}")

        return state

    def rewrite_tts_node(self, state: RadioState):
        self.debug(f"script: {state['script']}")

        prompt = self.REWRITE_TTS_PROMPT.replace("{LEVEL}", self.level).replace(
            "{SCRIPT}", state["script"]
        )

        self.debug(f"rewrite_tts_node prompt: {prompt}")

        tts_script = self.run_prompt(prompt, self.test_sectors)
        tts_script = self.replace_sectors(tts_script)

        state["tts_script"] = tts_script

        self.debug(f"generated tts_script: {tts_script}")

        return state

    def tts_node(self, state: RadioState):
        # Save the TTS script to a file
        now = state["now"]
        self.script_manager.save_script(state["tts_script"], state["topic"], now)

        tts_path = self.audio_manager.save_tts(state["tts_script"], now)
        self.debug(f"🎧 TTS audio saved for topic '{state['topic']}' at {tts_path}")

        audio_path = self.audio_manager.mix_bgm(now)
        self.debug(f"🎶 BGM mixed for topic '{state['topic']}' at {audio_path}")

        state["audio_path"] = audio_path

        return state

In [169]:
graph = Radiograph(
    client=client,
    base_path=Path("./jlpt_n4"),
    level="JLPT N4",
    sectors=["オープニング", "セグメント", "コーナー", "エンディング"],
    TOPIC_PROMPT=TOPIC_PROMPT,
    SCRIPT_PROMPT=SCRIPT_PROMPT,
    REWRITE_TTS_PROMPT=REWRITE_TTS_PROMPT,
    is_debug=True,
)
app = graph.compile()

In [170]:
graph.audio_manager.save_tts(
    "안녕하세요 저는 바보 바보 바보\n 나는 바보오~~ tts 테스트 중임을 알았으면 해요~~흐히 엔딩구~~",
    "asdf",
)

BadRequestError: Error code: 400 - {'error': {'message': 'Model gpt-4o-transcribe does not exist', 'code': 400}}

In [152]:
result = app.invoke(RadioState())

previous_topics: ['AAA', 'BBB', '주말에 가볼 곳', '오늘의 한국 음식 리뷰', '저녁 빠른 도시락 만들기', '간단한 점심 메뉴 고르기']
topic_node prompt: 
일본어 라디오에서 사용할 주제를 1개 생성하라.

[조건]
- JLPT JLPT N4 청취자가 이해할 수 있는 쉬운 주제
- 일상적이고 공감 가능한 내용
- 너무 추상적이거나 어려운 개념 금지

[출력 규칙]
- 오직 "주제 한 줄"만 출력
- 불필요한 설명, 따옴표, 번호, 줄바꿈 금지
- 20자 이내로 작성

[중복 방지]
- 아래 주제들과 의미적으로 겹치지 않도록 할 것:
["AAA", "BBB", "주말에 가볼 곳", "오늘의 한국 음식 리뷰", "저녁 빠른 도시락 만들기", "간단한 점심 메뉴 고르기"]

response: ChatCompletion(id='gen-1784349506-jpiS0zu5iRw1Hp4qcEIt', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='가벼운 커피 한잔 이야기', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user says: "일본어 라디오에서 사용할 주제를 1개 생성하라." That means "Generate one topic for use in a Japanese radio." They give constraints: JLPT N4 level listeners, everyday relatable content, no too abstract or difficult concepts. Output rule: only "topic one line", no unnecessary explanations, no quotes, numbe

BadRequestError: Error code: 400 - {'error': {'message': 'Model gpt-4o-mini-tts does not exist', 'code': 400}}

In [145]:
result

{'topic': '간단한 점심 메뉴 고르기',
 'script': '**オープニング**  \n[ゆるっと電波]  \nハヤト: গেলো যাঊ? হ্যালো, yo, আপনাকে স্বাগতম! আমি হায়াতো। আজকের পর্যবেক্ষণ? আপনি সহজে লাঞ্চ মেনু বেছে নিতে চান। তাই, আজকেও নরমভাবে কুলে শোনা দিই!  \n\n**[セグメント]**  \nহায়াতো: আজকের থিম: 简单な 점심 메뉴 고르기।  \nহায়াতো: রান্না এখন তাড়াতাড়ি করে, সাথে সাথে সুস্বাদু এমন খাবার দারুন।  \nহায়াতো: ভাবুন, ওনিগিরি, পাস্তা, কারি? সবই সঠিক বিকল্প!  \nহায়াতো: দিনগুলি কখনও কখনও তাড়াহুড়ো, লাঞ্চ সম্পর্কে আপনি কী ভাবেন?  \n\n**[セグメント]**  \nহায়াতো: আমি মনে করি, "বিয়তা" ফাইলটা ঠিক আছে।  \nহায়াতো: পারফেক্ট? ডোরিতে অবিধানে আসতে হবে।  \nহায়াতো: আজ সকালে কেমন ছিল? বেতালো, জাইফজং?  \n\n**[セグメント]**  \nহায়াতো: আপনার লাঞ্চ পরিকল্পনা শুনলাম!  \nহায়াতো: আপনি মিশা ভাবুন, "হাহা" বোলে কোনো দর্প্য বা?  \nহায়াতো: আপনিসহ অবিচার নেই, তাহলে এখন?  \n\n**猟】**  \nহায়াতো: আজকের গুরুত্বপূর্ণ শব্দ দেই!  \n\n**コーナー**  \n1. 「おすすめ」 (すすめろう)  \n   - অর্থ: পরামর্শ দিন, সুপারিশশীল খাদ্য হচ্ছে।  \n   - ব্যবহার: "明日、何を食べる？おすすめはありますか？"  \n\n2. 「ランチにする」  \n   - অর্থ: দ

In [146]:
print(result["tts_script"])

[オープニング]  
今日はゆるっと電波へようこそ！  
ゆっくりした音楽にのせて、まったりとお話ししますね。  
本日はランチメニューの選び方を楽しく考えてみましょう。  
それでは、簡単スタートです！  

[セグメント]  
ランチを選ぶときは、時間と味覚を考えます。  
おにぎりは手軽で、鮭や梅、塩味が好きです。  
パスタなら、トマトソースやクリームソースで簡単に調理できます。  
カレーはご飯と合わせると満足感が上がりますよ。  
どのメニューが好きか、皆さんはどうですか？  

[コーナー]  
「おすすめ」： 何かを紹介するときに使います。  
例： 明日、何を食べますか？ おすすめはありますか？  

「ランチにする」： ランチを決めるときに使います。  
例： 今日、何をランチにしますか？  

「ですか？」： 質問を柔らかくする言葉です。  
例： このメニューですか？  

[エンディング]  
今日の放送はここまでです。  
よいランチを楽しんでくださいね。 Stellung.  
 Fact, eh  Z   音、笑み感慨の中、またゆるっと電波へ！?
